In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

/Users/johnson/Documents/article-dev/fine-tuning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [3]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [5]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16
)
model.to(device)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 4373.82it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [7]:
model.config.use_cache = False  # Disable caching for training

In [8]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

In [9]:
# Training configuration
sft_config = SFTConfig(
    output_dir="outputs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_length=1024,
    packing=False,
    gradient_checkpointing=True,

    # turn these off on macOS
    fp16=False,
    bf16=False,

    optim="adamw_torch",
    dataloader_pin_memory=False,

    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",

    assistant_only_loss=True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/var/folders/4g/cnj2gf814xn3sbtcz0jrsb9w0000gn/T/ipykernel_27531/2332988166.py:2: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


In [11]:
# Load dataset
dataset = load_dataset(
    "json",
    data_files={"train": "../data/train.jsonl", "validation": "../data/val.jsonl"}
)

Generating train split: 240 examples [00:00, 56660.64 examples/s]
Generating validation split: 30 examples [00:00, 32742.42 examples/s]


In [13]:
# Build the trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    processing_class=tokenizer
)

Tokenizing eval dataset: 100%|██████████| 30/30 [00:00<00:00, 4155.79 examples/s]


In [14]:
trainer.model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [15]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.510148,0.343474,0.445801,0.899109,20147.000000
2,0.050407,0.093662,0.076843,0.974968,40294.000000
3,0.043560,0.085814,0.062836,0.978826,60441.000000


TrainOutput(global_step=90, training_loss=0.4670744844608837, metrics={'train_runtime': 370.6887, 'train_samples_per_second': 1.942, 'train_steps_per_second': 0.243, 'total_flos': 504974192578560.0, 'train_loss': 0.4670744844608837, 'epoch': 3.0})

In [17]:
trainer.save_model("outputs/final-adapter")
tokenizer.save_pretrained("outputs/final-adapter")
print("Done. Adapter saved to outputs/final-adapter")

Done. Adapter saved to outputs/final-adapter
